<h1>🔗 Biofilter — Report: <code>expand_entity_relationship</code></h1>

Which links the bundle holds for these entities.

One row per (input, relationship). Use it to answer *what is this
connected to*, or — with `relationship_scope="between_inputs"` — *how are
these connected to each other*.

### 1. Open a bundle

In [ ]:
from pathlib import Path

from biofilter import Biofilter

# Leave as None to use `[database] bundle` from .biofilter.toml.
BUNDLE = None
REPORT = "expand_entity_relationship"

bf = Biofilter(bundle=BUNDLE, debug_mode=False) if BUNDLE else Biofilter(debug_mode=False)

# Results land here whatever directory the kernel was started in.
_root = next(
    (p for p in [Path.cwd(), *Path.cwd().parents] if (p / ".biofilter.toml").is_file()),
    Path.cwd(),
)
OUTPUT_DIR = _root / "notebooks" / "templates" / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

bf

### 2. What the report offers

In [ ]:
print("columns:")
for column in bf.report.available_columns(REPORT):
    print(" ", column)

print("\nexample input:")
print(bf.report.example_input(REPORT))

In [ ]:
print(bf.report.explain(REPORT))

### 3. What is this connected to

The default scope, `input_to_any`, returns every link where an input
appears on either side. That is a lot: filter it.

In [ ]:
result = bf.report.run(
    REPORT,
    input_data=["TP53"],
    output_entity_groups=["Pathways"],
)
df = result.to_pandas()

print(f"{len(df):,} rows")
df[["input_original", "relationship_type", "related_primary_name",
    "related_group_name", "direction"]].head(10)

### 4. How are these connected to each other

`between_inputs` keeps only links whose **both** ends are in your list.
It is a different question, and usually a much smaller answer.

In [ ]:
genes = ["TP53", "BRCA1", "EGFR", "MDM2"]

for scope in ("input_to_any", "between_inputs"):
    out = bf.report.run(REPORT, input_data=genes, relationship_scope=scope).to_pandas()
    real = out[out["observation"] == ""]
    print(f"{scope:15s} {len(real):>7,} relationship rows")

In [ ]:
between = bf.report.run(
    REPORT, input_data=genes, relationship_scope="between_inputs"
).to_pandas()

between[between["observation"] == ""][
    ["input_primary_name", "relationship_type", "related_primary_name", "direction"]
].head(12)

### 5. Why a relationship can appear twice

A link with an input at **both** ends is reached from each, producing two
rows that differ in `match_side` and `direction`.

In `between_inputs` that always happens, so `deduplicate_pairs` defaults
to `True` there and `False` in `input_to_any`. Both are overridable.

In [ ]:
for dedupe in (True, False):
    out = bf.report.run(
        REPORT, input_data=genes,
        relationship_scope="between_inputs", deduplicate_pairs=dedupe,
    ).to_pandas()
    real = out[out["observation"] == ""]
    print(f"deduplicate_pairs={str(dedupe):5s} {len(real):>5,} rows")

### 6. Three kinds of row

| `observation` | meaning |
| --- | --- |
| *(empty)* | a real relationship |
| `not found` | the bundle has no entity for this input |
| `no relationships in scope` | it resolved, and nothing came back |

The third is new in 4.3.0. The report this replaces emitted rows only for
inputs it could not **resolve**, so a gene with five thousand
relationships and none to `Chemicals` simply vanished from a
chemicals-filtered result — indistinguishable from one never asked
about.

In [ ]:
mixed = bf.report.run(
    REPORT,
    input_data=["TP53", "GO:0006915", "ZZZ_NOT_A_THING"],
    output_entity_groups=["Chemicals"],
).to_pandas()

mixed[["input_original", "input_entity_id", "input_group_name", "observation"]]

### 7. Narrowing further

`input_entity_groups` constrains what an input may resolve to;
`relationship_types` keeps only certain kinds of link.

In [ ]:
print("relationship types in this result:")
print(bf.report.run(REPORT, input_data=["TP53"]).to_pandas()["relationship_type"]
      .value_counts().to_string())

In [ ]:
bf.report.run(
    REPORT,
    input_data=["TP53"],
    relationship_types=["in_pathway"],
    output_entity_groups=["Pathways"],
).to_pandas()[["input_original", "relationship_type", "related_primary_name"]].head(5)

### 8. Export

In [ ]:
for path in result.write(OUTPUT_DIR / "expand_entity_relationship.csv"):
    print(path)

### 9. The same thing on the command line

```bash
biofilter report run --report-name expand_entity_relationship \\
    --input TP53 --input BRCA1 \\
    --param relationship_scope=between_inputs \\
    --output relationships.csv
```

### 10. Quick QA

In [ ]:
expected = list(bf.report.available_columns(REPORT))
missing = [c for c in expected if c not in df.columns]

print("missing columns:", missing or "none")
print(df["observation"].value_counts(dropna=False).to_string())
print("bundle:", result.provenance["bundle_id"])
display(df.dtypes.to_frame("dtype"))